# Linear ridge regression example for the challenge Dyni Odontocete Click Classification, 10 species

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
import time

In [ ]:
# read data
X_train = np.load('./DOCC10_train/DOCC10_Xtrain.npy')
X_test = np.load('./DOCC10_test/DOCC10_Xtest.npy')
Y_train_df = pd.read_csv('./DOCC10_Ytrain.csv', index_col=0)


print('X train', X_train.shape, 'Y_train', Y_train_df.shape, 'X_test', X_test.shape)
Y_train_df.describe()

In [ ]:
# encorde string labels with one-hot vectors
Y_tr_ = Y_train_df['TARGET'].values.reshape(-1, 1)
enc = OneHotEncoder()
enc.fit(Y_tr_)
Y_tr = enc.transform(Y_tr_).toarray()

In [ ]:
# shuffle train set
perm = np.random.permutation(X_train.shape[0])
X = X_train[perm]
y = Y_tr[perm]
X.shape, y.shape

In [ ]:
# cross validate the ridge parameter alpha
alphas = 10.**(-np.arange(-5,5))
best_acc = 0
best_alpha = -1

for alpha in alphas:
    t0 = time.time()
    reg = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    y_pred = cross_val_predict(reg, X=X, y=y, cv=4)
    acc = accuracy_score(np.argmax(y, axis=1), np.argmax(y_pred, axis=1))
    if acc > best_acc:
        best_acc, best_alpha = acc, alpha
    t = (time.time() - t0) / 60
    print(f'alpha={alpha} acc={acc}, t = {t:.1f} min')

In [ ]:
# fit regressor on the whole train db
reg = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
reg.fit(X, y)

In [ ]:
# save predictions to csv for submission
y_test_pred = reg.predict(X_test.astype('float32'))
y_test_pred = (y_test_pred == y_test_pred.max(axis=1, keepdims=True)).astype(float)
y_test_pred = enc.inverse_transform(y_test_pred)
Y_test_pred_df = pd.DataFrame(data=y_test_pred, index=np.arange(y_test_pred.shape[0]), columns=['TARGET'])
Y_test_pred_df.to_csv('./DOCC10_Ytest_pred_linear.csv', index_label='ID')